#Imports

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#Load Dataset

In [3]:
path = "/content/roommate_compatibility_dataset_v2.csv"
df = pd.read_csv(path)
df.head()

,sleep_schedule_diff,wakeup_schedule_diff,cleanliness_diff,study_hours_diff,noise_tolerance_diff,room_temperature_diff,social_energy_diff,spending_habit_diff,guest_frequency_diff,gaming_time_diff,...,messy_vs_clean_conflict,nightowl_vs_earlybird_conflict,loud_music_vs_noise_sensitive,guest_heavy_vs_private_person,high_spender_vs_budget_person,group_study_vs_solo_study,fitness_focused_vs_irregular,emotional_expressive_vs_reserved,compatibility_score,compatibility_label
0,0.88,0.27,0.28,0.50,0.77,0.06,0.85,0.17,0.42,0.37,...,1,1,1,1,0,1,0,1,1.54,1
1,0.57,0.14,0.33,0.86,0.39,0.32,0.66,0.46,0.05,0.48,...,1,0,0,1,1,1,1,1,47.86,3
2,0.32,0.14,0.57,0.18,0.21,0.12,0.57,0.21,0.52,0.71,...,0,1,0,1,1,0,0,0,30.62,2
3,0.25,0.34,0.74,0.05,0.73,0.25,0.17,0.19,0.57,0.46,...,0,1,1,1,0,1,1,0,56.28,3
4,0.24,0.42,0.84,0.53,0.46,0.10,0.69,0.01,0.22,0.47,...,0,1,0,1,0,0,0,0,37.62,2


In [4]:
X = df.drop(columns=["compatibility_score","compatibility_label"])
y = df["compatibility_label"]

In [5]:
print(X.shape)
print(y.shape)

(30000, 30)
(30000,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(24000, 30)
(6000, 30)


In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Calculate Importance using XGBCLassifier

In [8]:
xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

xgb_preds = xgb_model.predict(X_test_scaled)

In [9]:
mae = mean_absolute_error(y_test, xgb_preds)

rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

r2 = r2_score(y_test, xgb_preds)

print("XGBOOST RESULTS")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

XGBOOST RESULTS
MAE : 0.41439682245254517
RMSE: 0.5451628176248985
R2  : 0.7959456443786621


In [10]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="importance",
    ascending=False
)

importance_df.head(15)

,feature,importance
22,messy_vs_clean_conflict,0.231664
23,nightowl_vs_earlybird_conflict,0.170971
20,gamer_vs_light_sleeper,0.146209
24,loud_music_vs_noise_sensitive,0.128962
25,guest_heavy_vs_private_person,0.082136
21,extrovert_vs_introvert_conflict,0.077753
27,group_study_vs_solo_study,0.015007
29,emotional_expressive_vs_reserved,0.014502
26,high_spender_vs_budget_person,0.014274
17,academic_goal_similarity,0.013861


In [11]:
TOP_K = 20

top_features = importance_df.head(TOP_K)["feature"].tolist()

print(top_features)

['messy_vs_clean_conflict', 'nightowl_vs_earlybird_conflict', 'gamer_vs_light_sleeper', 'loud_music_vs_noise_sensitive', 'guest_heavy_vs_private_person', 'extrovert_vs_introvert_conflict', 'group_study_vs_solo_study', 'emotional_expressive_vs_reserved', 'high_spender_vs_budget_person', 'academic_goal_similarity', 'study_style_similarity', 'fitness_focused_vs_irregular', 'communication_style_similarity', 'organization_similarity', 'food_preference_similarity', 'hobby_similarity', 'weekend_activity_similarity', 'cleanliness_diff', 'social_energy_diff', 'noise_tolerance_diff']


In [12]:
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

scaler2 = StandardScaler()

X_train_top = scaler2.fit_transform(X_train_top)
X_test_top = scaler2.transform(X_test_top)

#pyTorch Dataset


In [13]:
class RoommateDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)

        self.y = torch.tensor(
            y.values,
            dtype=torch.float32
        ).view(-1, 1)

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return self.X[idx], self.y[idx]

In [14]:
train_dataset = RoommateDataset(
    X_train_top,
    y_train
)

test_dataset = RoommateDataset(
    X_test_top,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

#Model

In [15]:
class MLPRegressor(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):

        return self.network(x)

In [16]:
model = MLPRegressor(
    input_size=TOP_K
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [17]:
epochs = 20
best_loss = float('inf')
for epoch in range(epochs):
  model.train()
  train_loss = 0
  for x_batch,y_batch in train_loader:
    x_batch,y_batch = x_batch.to(device),y_batch.to(device)
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss = train_loss / len(train_loader)

  model.eval()
  with torch.no_grad():
    val_loss = 0
    for x_batch,y_batch in test_loader:
      x_batch,y_batch = x_batch.to(device),y_batch.to(device)
      preds = model(x_batch)
      loss = criterion(preds,y_batch)
      val_loss += loss
    val_loss = val_loss / len(test_loader)
    if val_loss < best_loss:
      best_loss = val_loss
      torch.save(model.state_dict(),'best_model.pth')

    print(f'Epoch : {epoch+1}/{epochs} | Train_Loss : {train_loss:.2f} | Val_Loss : {val_loss:.2f}')



Epoch : 1/20 | Train_Loss : 1.06 | Val_Loss : 0.37
Epoch : 2/20 | Train_Loss : 0.47 | Val_Loss : 0.35
Epoch : 3/20 | Train_Loss : 0.44 | Val_Loss : 0.34
Epoch : 4/20 | Train_Loss : 0.42 | Val_Loss : 0.33
Epoch : 5/20 | Train_Loss : 0.41 | Val_Loss : 0.34
Epoch : 6/20 | Train_Loss : 0.39 | Val_Loss : 0.32
Epoch : 7/20 | Train_Loss : 0.38 | Val_Loss : 0.33
Epoch : 8/20 | Train_Loss : 0.38 | Val_Loss : 0.33
Epoch : 9/20 | Train_Loss : 0.37 | Val_Loss : 0.33
Epoch : 10/20 | Train_Loss : 0.36 | Val_Loss : 0.32
Epoch : 11/20 | Train_Loss : 0.36 | Val_Loss : 0.34
Epoch : 12/20 | Train_Loss : 0.35 | Val_Loss : 0.32
Epoch : 13/20 | Train_Loss : 0.35 | Val_Loss : 0.37
Epoch : 14/20 | Train_Loss : 0.35 | Val_Loss : 0.37
Epoch : 15/20 | Train_Loss : 0.35 | Val_Loss : 0.36
Epoch : 16/20 | Train_Loss : 0.35 | Val_Loss : 0.34
Epoch : 17/20 | Train_Loss : 0.34 | Val_Loss : 0.33
Epoch : 18/20 | Train_Loss : 0.34 | Val_Loss : 0.33
Epoch : 19/20 | Train_Loss : 0.34 | Val_Loss : 0.36
Epoch : 20/20 | Train

In [24]:
model = MLPRegressor(
    input_size=TOP_K
).to(device)
model.load_state_dict(torch.load('best_model.pth'))


<All keys matched successfully>

In [25]:
epochs = 20
best_loss = float('inf')
for epoch in range(epochs):
  model.train()
  train_loss = 0
  for x_batch,y_batch in train_loader:
    x_batch,y_batch = x_batch.to(device),y_batch.to(device)
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()
  train_loss = train_loss / len(train_loader)

  model.eval()
  with torch.no_grad():
    val_loss = 0
    for x_batch,y_batch in test_loader:
      x_batch,y_batch = x_batch.to(device),y_batch.to(device)
      preds = model(x_batch)
      loss = criterion(preds,y_batch)
      val_loss += loss
    val_loss = val_loss / len(test_loader)
    if val_loss < best_loss:
      best_loss = val_loss
      torch.save(model.state_dict(),'best_model.pth')

    print(f'Phase 2 Epoch : {epoch+1}/{epochs} | Train_Loss : {train_loss:.2f} | Val_Loss : {val_loss:.2f}')



Phase 2 Epoch : 1/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 2/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 3/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 4/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 5/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 6/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 7/20 | Train_Loss : 0.40 | Val_Loss : 0.32
Phase 2 Epoch : 8/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 9/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 10/20 | Train_Loss : 0.40 | Val_Loss : 0.32
Phase 2 Epoch : 11/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 12/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 13/20 | Train_Loss : 0.40 | Val_Loss : 0.32
Phase 2 Epoch : 14/20 | Train_Loss : 0.41 | Val_Loss : 0.32
Phase 2 Epoch : 15/20 | Train_Loss : 0.40 | Val_Loss : 0.32
Phase 2 Epoch : 16/20 | Train_Loss : 0.40 | Val_Loss : 0.32
Phase 2 Epoch : 17/20 | Train_Loss : 0.40 | Val_L

In [26]:
model = MLPRegressor(
    input_size=TOP_K
).to(device)
model.load_state_dict(torch.load('best_model.pth'))


<All keys matched successfully>

In [27]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        predictions.extend(
            outputs.cpu().numpy().flatten()
        )

        actuals.extend(
            y_batch.numpy().flatten()
        )

In [28]:
mae = mean_absolute_error(actuals, predictions)

rmse = np.sqrt(
    mean_squared_error(actuals, predictions)
)

r2 = r2_score(actuals, predictions)

print("\nMLP RESULTS")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)


MLP RESULTS
MAE : 0.42221879452466965
RMSE: 0.5616842138146915
R2  : 0.7833902847956742


In [30]:
def score_to_label(score):

    if score <= 20:
        return 1

    elif score <= 40:
        return 2

    elif score <= 60:
        return 3

    elif score <= 80:
        return 4

    else:
        return 5

In [39]:
sample = X_test.iloc[1020]

sample_df = pd.DataFrame([sample])

sample_df = sample_df[top_features]

sample_scaled = scaler2.transform(sample_df)

sample_tensor = torch.tensor(
    sample_scaled,
    dtype=torch.float32
).to(device)

model.eval()

with torch.no_grad():

    pred_score = model(sample_tensor).item()

pred_score = max(0, min(100, pred_score))

pred_label = score_to_label(pred_score)

print("Predicted Score :", round(pred_score, 2))
print("Predicted Label :", pred_label)
print("Actual Label :",y_test.iloc[1020])


Predicted Score : 4.56
Predicted Label : 1
Actual Label : 4


In [43]:
epochs = 10
for epoch in range(epochs):
  actual =[]
  pred = []
  model.train()

  for x_batch,y_batch in train_loader:
    x_batch,y_batch = x_batch.to(device),y_batch.to(device)
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)
    loss.backward()
    optimizer.step()
    actual.extend(y_batch.cpu().numpy().flatten())
    pred.extend(preds.detach().cpu().numpy().flatten())
  model.eval()
  train_r2 = r2_score(actual,pred)
  actual =[]
  pred = []

  with torch.no_grad():
    for x_batch,y_batch in test_loader:
      x_batch,y_batch = x_batch.to(device),y_batch.to(device)
      preds = model(x_batch)
      loss = criterion(preds,y_batch)
      actual.extend(y_batch.cpu().numpy().flatten())
      pred.extend(preds.cpu().numpy().flatten())
    val_r2 = r2_score(actual,pred)
    print(f'Epoch : {epoch+1}/{epochs} | Train_R2 : {train_r2:.2f} | Val_R2 : {val_r2:.2f}')



Epoch : 1/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 2/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 3/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 4/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 5/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 6/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 7/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 8/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 9/10 | Train_R2 : 0.72 | Val_R2 : 0.78
Epoch : 10/10 | Train_R2 : 0.72 | Val_R2 : 0.78


In [44]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0003
)

In [45]:
epochs = 60
for epoch in range(epochs):
  actual =[]
  pred = []
  model.train()

  for x_batch,y_batch in train_loader:
    x_batch,y_batch = x_batch.to(device),y_batch.to(device)
    optimizer.zero_grad()
    preds = model(x_batch)
    loss = criterion(preds,y_batch)
    loss.backward()
    optimizer.step()
    actual.extend(y_batch.cpu().numpy().flatten())
    pred.extend(preds.detach().cpu().numpy().flatten())
  model.eval()
  train_r2 = r2_score(actual,pred)
  actual =[]
  pred = []

  with torch.no_grad():
    for x_batch,y_batch in test_loader:
      x_batch,y_batch = x_batch.to(device),y_batch.to(device)
      preds = model(x_batch)
      loss = criterion(preds,y_batch)
      actual.extend(y_batch.cpu().numpy().flatten())
      pred.extend(preds.cpu().numpy().flatten())
    val_r2 = r2_score(actual,pred)
    print(f'Epoch : {epoch+1}/{epochs} | Train_R2 : {train_r2:.2f} | Val_R2 : {val_r2:.2f}')



Epoch : 1/60 | Train_R2 : 0.74 | Val_R2 : 0.77
Epoch : 2/60 | Train_R2 : 0.75 | Val_R2 : 0.75
Epoch : 3/60 | Train_R2 : 0.75 | Val_R2 : 0.77
Epoch : 4/60 | Train_R2 : 0.75 | Val_R2 : 0.77
Epoch : 5/60 | Train_R2 : 0.75 | Val_R2 : 0.78
Epoch : 6/60 | Train_R2 : 0.75 | Val_R2 : 0.78
Epoch : 7/60 | Train_R2 : 0.76 | Val_R2 : 0.78
Epoch : 8/60 | Train_R2 : 0.76 | Val_R2 : 0.78
Epoch : 9/60 | Train_R2 : 0.76 | Val_R2 : 0.77
Epoch : 10/60 | Train_R2 : 0.76 | Val_R2 : 0.77
Epoch : 11/60 | Train_R2 : 0.76 | Val_R2 : 0.75
Epoch : 12/60 | Train_R2 : 0.76 | Val_R2 : 0.78
Epoch : 13/60 | Train_R2 : 0.76 | Val_R2 : 0.76
Epoch : 14/60 | Train_R2 : 0.76 | Val_R2 : 0.78
Epoch : 15/60 | Train_R2 : 0.76 | Val_R2 : 0.76
Epoch : 16/60 | Train_R2 : 0.76 | Val_R2 : 0.78
Epoch : 17/60 | Train_R2 : 0.76 | Val_R2 : 0.77
Epoch : 18/60 | Train_R2 : 0.76 | Val_R2 : 0.77
Epoch : 19/60 | Train_R2 : 0.77 | Val_R2 : 0.76
Epoch : 20/60 | Train_R2 : 0.77 | Val_R2 : 0.77
Epoch : 21/60 | Train_R2 : 0.77 | Val_R2 : 0.78
E